# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shwetabh1013/flyrank-ml-internship-shwetabh/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

Shared helpers live in `work/refresh_lib.py` (loading, the two feature sets, the W04 rule, the split generator) so W06, W07 and the capstone all run the *same* logic. Everything that matters for a reader is explained here, in the notebook.

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

sys.path.append(str(Path("..").resolve()))      # work/
import refresh_lib as rl
from refresh_lib import precision_at_k

warnings.filterwarnings("ignore", category=FutureWarning)
OUT = Path("../outputs"); OUT.mkdir(exist_ok=True)

df = rl.load_data()
print(f"{len(df):,} rows | {df['client_id'].nunique()} clients | declining rate {df['y'].mean():.3f}")

30,000 rows | 32 clients | declining rate 0.542


## 1. Two paper findings + my methodology questions

Source: *The State of AI-Driven SEO, March 2026* (`docs/flyrank-seo-research-march-2026.pdf`). Both findings below touch my lane directly, because they use the same `trend_direction` idea my label comes from. The goal is "how to make it stronger", not a gotcha — the paper itself already labels its ML pages as exploratory.

### Finding A — "The Anatomy of Growing Content" (Finding #1, CONFIRMED)
**Claim:** growing pages are longer (3.2K vs 2.3K words) and younger (184 vs 230 days) than declining pages.

- **Where does the label come from?** `trend_direction`: last-30-day vs previous-30-day impressions, with a ±10% band. A page going from 2 impressions to 1 is "down" exactly like a page going from 20,000 to 10,000. The paper doesn't say whether a minimum-traffic floor was applied.
- **Does the design carry the claim?** As written — "directionally robust ... observational" — mostly yes. The comparison is cross-sectional, so it can't say that *adding* words prevents decline (the paper's "What to do" section edges toward that). Age and length are also confounded (older content was written under older length norms), which the paper itself lists under confounders.
- **Stronger version:** apply a traffic floor to the label, and report the gap within age tiers so length and age aren't tangled.

### Finding B — ML appendix: "What Predicts Growth?" (logistic regression, 71% holdout accuracy)
- **Where does the label come from?** The same growing-vs-declining split. In the paper's own table that is 74.8K up vs 45.6K down, so **always guessing "up" scores ~62%**. 71% is ~9 points of skill, not 71 — the base rate should sit next to it.
- **Does the design carry the claim?** The split is "80/20" with no mention of grouping by brand. With 57 brands, a random split lets the model learn each brand's typical trend and recognise it again in the test set. Features listed include "recent impressions" and "days visible" — if those are trailing-90-day totals they overlap the 30-day label window.
- **Stronger version:** brand-grouped split, base rate printed beside accuracy, and features restricted to the window *before* the label.

The code below checks whether Finding A reproduces in my slice, and computes the base-rate point for Finding B on my data. Section 2 then measures how much a random split inflates scores — the same risk Finding B carries.

In [2]:
up_down = df[df["trend_direction"].isin(["up", "down"])]
summary = up_down.groupby("trend_direction").agg(
    pages=("content_id", "size"),
    median_words=("word_count", "median"),
    median_age_days=("content_age_days", "median"),
)
print("Finding A, re-checked on my slice (all rows):")
print(summary, "\n")

m = rl.measurable(up_down)
print(f"Same check with a traffic floor (>= {rl.MEASURABLE_FLOOR} impressions in days 31-60):")
print(m.groupby("trend_direction").agg(pages=("content_id", "size"),
      median_words=("word_count", "median"), median_age_days=("content_age_days", "median")), "\n")

share_up = (up_down["trend_direction"] == "up").mean()
print(f"Finding B base rate in my slice: always guessing the majority class = {max(share_up, 1-share_up):.1%}")
print("Paper's own counts: 74.8K up vs 45.6K down -> majority guess = "
      f"{74.8/(74.8+45.6):.1%}, so 71% accuracy is ~{71-100*74.8/(74.8+45.6):.0f} points above it")

Finding A, re-checked on my slice (all rows):
                 pages  median_words  median_age_days
trend_direction                                      
down             16262        2909.0            216.0
up                4388        2847.5            291.5 

Same check with a traffic floor (>= 30 impressions in days 31-60):
                 pages  median_words  median_age_days
trend_direction                                      
down             13266        2936.0            216.0
up                3103        2900.0            300.0 

Finding B base rate in my slice: always guessing the majority class = 78.8%
Paper's own counts: 74.8K up vs 45.6K down -> majority guess = 62.1%, so 71% accuracy is ~9 points above it


**What this shows:**
- **Finding A does not reproduce in my slice — partly reversed.** Declining pages here are *younger* (median 216 days vs 291 for growing pages, same with the traffic floor), and word counts are nearly identical (~2.9K both). My slice is 30k pages from 32 clients, all ≥ 90 days old, so this is a scoping note, not a refutation: the paper's age gap may depend on the very young pages my slice doesn't contain. It does show the age pattern is population-dependent, which is a reason to report it within age tiers.
- **Finding B's base rate matters even more here:** among up/down pages in my slice, always guessing the majority class already scores 78.8%. Any accuracy number on this label is meaningless without that next to it — which is why I use precision@K against the base rate, never raw accuracy.

## 2. My model under an honest split (before/after)

My W05 headline (Random Forest precision@20 = 0.600) came from **one** client holdout: 6 test clients, 2,325 rows, and a test base rate of 0.391 — very different from the 0.542 overall. precision@20 on a single split is 20 pages from one draw; it can swing a lot.

So the "after" here is:
- **GroupKFold by client, 5 folds × 5 shuffles = 25 held-out client groups.** Every client is tested several times, never trained on in the same fold. Mean ± std, not one number.
- **Random row split (5×5)** alongside, *only* to measure how much memorising clients inflates the score. The gap is itself a finding.

Same W05 features, same Random Forest settings, nothing else changed.

In [3]:
rows = []
for kind in ["random", "grouped"]:
    X = rl.w05_features(df); y = df["y"].to_numpy()
    for seed, fold, tr, te in rl.splits(df, kind):
        if kind == "grouped":
            assert set(df.iloc[tr]["client_id"]).isdisjoint(df.iloc[te]["client_id"]), "client leak!"
        p = rl.make_rf().fit(X.iloc[tr], y[tr]).predict_proba(X.iloc[te])[:, 1]
        rows.append(dict(split=kind, p20=precision_at_k(y[te], p, 20),
                         p50=precision_at_k(y[te], p, 50),
                         auc=roc_auc_score(y[te], p), base_rate=y[te].mean()))
split_compare = pd.DataFrame(rows)
split_compare.to_csv(OUT / "w06_split_compare.csv", index=False)
before_after = split_compare.groupby("split").agg(["mean", "std"]).round(3)
before_after.loc["W05 single holdout (reported)"] = [0.600, np.nan, 0.600, np.nan, 0.735, np.nan, 0.391, np.nan]
before_after

p20           p50           auc         \
                                mean    std   mean    std   mean    std   
split                                                                     
grouped                        0.728  0.184  0.745  0.135  0.682  0.083   
random                         0.982  0.028  0.964  0.023  0.758  0.006   
W05 single holdout (reported)  0.600    NaN  0.600    NaN  0.735    NaN   

                              base_rate         
                                   mean    std  
split                                           
grouped                           0.533  0.084  
random                            0.542  0.006  
W05 single holdout (reported)     0.391    NaN

**Read:**
- **Random row split → near-perfect top of queue** (precision@20 ≈ 0.98). That is the model recognising clients it already saw, not skill. If I had used a random split in W05 I'd have reported a number that collapses on any new client.
- **Grouped split → the honest range.** Mean precision@20 is *higher* than my single-split 0.600, but the std is large (± ~0.18): individual held-out groups land anywhere from poor to excellent. My 0.600 was one draw from a wide distribution — it was never a stable estimate, high or low.
- AUC drops from ~0.76 (random) to ~0.68 (grouped). Section 3 asks whether even 0.68 is real.

## 3. Leakage audit

The Week 3 hunt (label-derived columns, product flags, overlapping windows), run on my final feature set.

### 3a. Label-derived columns and product metadata
`trend_direction`, `trend_pct` (label source), the target, ids, and `provider_used` / `model_used` (generation metadata the dictionary says is not a model feature) must not appear.

In [4]:
w05_cols = set(rl.W05_NUMERIC + rl.W05_CATEGORICAL)
print("banned columns in W05 feature list:", w05_cols & rl.BANNED or "none")
assert not (w05_cols & rl.BANNED)

banned columns in W05 feature list: none


Clean on this check.

### 3b. Overlapping windows — the one I missed in W05
The label compares **days 1–30** with **days 31–60** of impressions. Now look at what W05 fed the model:

| W05 feature | window | overlaps the label? |
|---|---|---|
| `impressions_90d`, `clicks_90d`, `ctr` | days 1–90 | **yes** — contains days 1–30, the outcome itself |
| `avg_position` (and `position_tier`) | days 1–90 mean | **yes** — a page that fell in days 1–30 also has a worse average position |
| `engagement_rate`, `scroll_rate` | days 1–90 | yes (weaker link) |
| `days_since_last_update` (+ `freshness_tier`) | as of export | **yes** — 68% of pages were updated *inside* days 1–30, i.e. during the outcome window |
| `content_age_days`, `word_count`, `content_type`, `main_intent` | static | no |

So W05 is not "predicting a decline". It is **describing a decline that already happened inside its own feature window.** That's fine for a dashboard, not for a forecast an editor acts on.

**Fix:** pretend the decision is made 30 days ago (T-30). Only days 31–90 plus static metadata are allowed. The dataset already splits impressions/clicks/sessions into last-30 / prev-30 / 90-day, so days 61–90 = 90d − last30 − prev30. Position, engagement and update dates have no sub-window in this CSV, so they're dropped (the warehouse's daily table would allow them — future work).

In [5]:
print("update timing:", f"{(df['days_since_last_update'] < 30).mean():.1%} of pages updated inside the label window")
print("top values of days_since_last_update:")
print(df["days_since_last_update"].value_counts().head(3).to_string(), "\n")

Xh = rl.honest_features(df, "T-30")
print("honest feature set:", list(Xh.columns))
assert not (set(Xh.columns) & rl.BANNED)

update timing: 68.3% of pages updated inside the label window
top values of days_since_last_update:
days_since_last_update
20     11573
104     8773
22      3564 

honest feature set: ['log_impressions_recent', 'log_impressions_before', 'impression_momentum', 'log_clicks_recent', 'log_clicks_before', 'ctr_60d', 'log_sessions_recent', 'log_sessions_before', 'content_age_days', 'word_count', 'has_word_count', 'search_volume', 'competition', 'cpc', 'has_keyword_data', 'content_type_comparison article', 'content_type_feedly article', 'content_type_keyword article', 'main_intent_commercial', 'main_intent_informational', 'main_intent_navigational', 'main_intent_transactional', 'main_intent_unknown']


### 3c. Harness check — plant a leak on purpose
If my evaluation code works, adding the one column that literally contains the outcome (`impressions_last_30d`) must send the score to ~1.0. If it doesn't, the harness is broken and none of my numbers mean anything.

In [6]:
def grouped_eval(frame, X, model_fn, label):
    y = frame["y"].to_numpy(); out = []
    for seed, fold, tr, te in rl.splits(frame, "grouped"):
        p = model_fn().fit(X.iloc[tr], y[tr]).predict_proba(X.iloc[te])[:, 1]
        out.append(dict(model=label, seed=seed, fold=fold, p20=precision_at_k(y[te], p, 20),
                        p50=precision_at_k(y[te], p, 50), auc=roc_auc_score(y[te], p),
                        base_rate=y[te].mean()))
    return pd.DataFrame(out)

planted = Xh.assign(log_impressions_last_30d=np.log1p(df["impressions_last_30d"]))
leak_check = pd.concat([
    grouped_eval(df, Xh, rl.make_rf, "honest features"),
    grouped_eval(df, planted, rl.make_rf, "honest + PLANTED leak"),
])
leak_check.groupby("model")[["p20", "p50", "auc"]].mean().round(3)

,p20,p50,auc
model,,,
honest + PLANTED leak,1.00,1.000,0.873
honest features,0.75,0.735,0.728


The planted leak jumps precision@20 to 1.000 — the harness catches leaks. Removed again; it never goes near a real model.

### 3d. Label noise floor
In W05, all three most-confidently-wrong pages had **1–3 impressions in 90 days**. A page going 1 → 0 impressions is "down" by definition, and nothing can or should predict that. I restrict the label to pages with **≥ 30 impressions in days 31–60** (the label's denominator; about one a day). This floor only uses information available at T-30, so it is legal at decision time too. It also removes a trivial pattern: pages with 0 impressions in days 31–60 can never be labelled "down", which would let a model score points for nothing.

In [7]:
m = rl.measurable(df)
print(f"measurable pages: {len(m):,} of {len(df):,} ({len(m)/len(df):.0%}), "
      f"{m['client_id'].nunique()} clients, declining rate {m['y'].mean():.3f}")
print(f"pages with 0 impressions in days 31-60 that are labelled 'down': "
      f"{((df['impressions_prev_30d'] == 0) & (df['y'] == 1)).sum()}  (always 0 -> trivially predictable)")

measurable pages: 21,578 of 30,000 (72%), 31 clients, declining rate 0.615
pages with 0 impressions in days 31-60 that are labelled 'down': 0  (always 0 -> trivially predictable)


### 3e. Final grouped comparison — every candidate on the same 25 folds
Measurable pages only. Same folds for everything (same seeds), so differences are model differences, not split luck.

- **base rate / random** — the floor any ranking has to clear
- **staleness only** — the W02 baseline
- **W04 rule** — benchmark fit on train clients only
- **momentum** — "pages that just spiked will fall back" (an honest single-feature baseline)
- **RF on W05 features** — kept for reference: shows what the overlapping windows were worth
- **RF / LR on honest features** — the candidates

**Selection rule, fixed before looking:** the deployable model is whichever honest-feature model has the higher **mean precision@20** (my primary metric since W01).

In [8]:
m = rl.measurable(df)
y = m["y"].to_numpy()
X_w05, X_hon = rl.w05_features(m), rl.honest_features(m, "T-30")
rows = []
for seed, fold, tr, te in rl.splits(m, "grouped"):
    train, test = m.iloc[tr], m.iloc[te]
    rng = np.random.default_rng(1000 * seed + fold)
    scores = {
        "random ranking": rng.random(len(test)),
        "staleness only": test["days_since_last_update"].to_numpy() + rng.random(len(test)) * 1e-3,
        "W04 rule": rl.w04_rule_score(train, test) + rng.random(len(test)) * 1e-6,
        "momentum": X_hon.iloc[te]["impression_momentum"].to_numpy(),
        "RF, W05 features (overlapping windows)":
            rl.make_rf().fit(X_w05.iloc[tr], y[tr]).predict_proba(X_w05.iloc[te])[:, 1],
        "RF, honest features": rl.make_rf().fit(X_hon.iloc[tr], y[tr]).predict_proba(X_hon.iloc[te])[:, 1],
        "LR, honest features": rl.make_lr().fit(X_hon.iloc[tr], y[tr]).predict_proba(X_hon.iloc[te])[:, 1],
    }
    for name, s in scores.items():
        rows.append(dict(model=name, seed=seed, fold=fold,
                         p20=precision_at_k(y[te], s, 20), p50=precision_at_k(y[te], s, 50),
                         auc=roc_auc_score(y[te], s),
                         client_p10=rl.per_client_precision(test, s, 10),
                         base_rate=y[te].mean(), n_test=len(te)))
folds = pd.DataFrame(rows)
folds.to_csv(OUT / "w06_fold_results.csv", index=False)

order = ["random ranking", "staleness only", "W04 rule", "momentum",
         "RF, W05 features (overlapping windows)", "RF, honest features", "LR, honest features"]
final_table = (folds.groupby("model")[["p20", "p50", "client_p10", "auc", "base_rate"]]
               .agg(["mean", "std"]).round(3).loc[order])
final_table

p20           p50        client_p10  \
                                         mean    std   mean    std       mean   
model                                                                           
random ranking                          0.614  0.156  0.622  0.113      0.605   
staleness only                          0.702  0.166  0.702  0.180      0.617   
W04 rule                                0.582  0.138  0.586  0.177      0.618   
momentum                                0.626  0.138  0.652  0.080      0.612   
RF, W05 features (overlapping windows)  0.844  0.141  0.864  0.100      0.828   
RF, honest features                     0.746  0.194  0.774  0.155      0.730   
LR, honest features                     0.828  0.091  0.818  0.083      0.758   

                                                 auc        base_rate         
                                          std   mean    std      mean    std  
model                                                                         
random ranking                          0.092  0.495  0.011     0.633  0.096  
staleness only                          0.105  0.489  0.066     0.633  0.096  
W04 rule                                0.105  0.522  0.042     0.633  0.096  
momentum                                0.087  0.481  0.052     0.633  0.096  
RF, W05 features (overlapping windows)  0.076  0.662  0.044     0.633  0.096  
RF, honest features                     0.090  0.625  0.037     0.633  0.096  
LR, honest features                     0.092  0.616  0.048     0.633  0.096

In [9]:
wide = folds.pivot_table(index=["seed", "fold"], columns="model", values="p20")
honest = ["RF, honest features", "LR, honest features"]
chosen = max(honest, key=lambda c: wide[c].mean())
print(f"Selected by mean precision@20: {chosen}")
for other in ["W04 rule", "staleness only", "random ranking"]:
    print(f"  {chosen} >= {other} on {(wide[chosen] >= wide[other]).mean():.0%} of 25 folds")
print(f"  {chosen} > fold base rate on "
      f"{(folds[folds.model == chosen].set_index(['seed','fold'])['p20'] > folds[folds.model == chosen].set_index(['seed','fold'])['base_rate']).mean():.0%} of folds")

Selected by mean precision@20: LR, honest features
  LR, honest features >= W04 rule on 96% of 25 folds
  LR, honest features >= staleness only on 84% of 25 folds
  LR, honest features >= random ranking on 96% of 25 folds
  LR, honest features > fold base rate on 100% of folds


**Read:**
1. **The W04 rule is at or below the base rate** on held-out clients (AUC ≈ 0.52, precision@20 below the ~0.63 base rate). That confirms W05's finding with 25 folds instead of one. The stale-AND-CTR-gap idea doesn't transfer.
2. **Staleness-only and momentum have AUC ≤ 0.50** — no ranking signal. Staleness is mostly two batch-update dates (20 and 104 days), not real editorial history.
3. **The overlapping-window RF scores best** — and that is exactly the problem. Its edge over the honest models (~0.04 AUC) is roughly what reading the outcome window was worth.
4. **The honest models still clear the bar.** The selected model lifts the top of the queue from the ~0.63 base rate to ~0.83 precision@20, and wins against the W04 rule on most folds. Overall discrimination is modest (AUC ≈ 0.62): **the model is useful for picking a short review list, not for ranking the whole site.**
5. Logistic regression is the steadier performer at the top of the queue (smaller std) and its coefficients are readable — which matters for reason codes in W07.

## 4. Claim rewrite

**My boldest sentence (W05, section 4):**
> "Did any model beat the W04 rule? Yes, clearly. Random Forest scores precision@20 = 0.600 and precision@50 = 0.600 — well above the W04 rule's 0.300 / 0.180."

**What's wrong with it:** one split of 6 clients; precision@20 is 20 pages; features overlapped the label's window, so the model partly read the outcome; "clearly" is a feeling, not a measurement.

**Rewritten (observed / measured / directional / decision-support):**
> "On 25 client-grouped folds (pages with ≥ 30 impressions in the comparison month), a logistic regression that sees only information available 30 days before the outcome window flagged declining pages at a measured mean precision@20 of ~0.83 (± ~0.09), against a ~0.63 base rate and ~0.58 for the hand-built W04 rule. Overall ranking quality is modest (AUC ≈ 0.62), so we use it as decision support for choosing which pages a person reviews first — not as a forecast of traffic, and not as evidence that refreshing a page causes recovery."

The code prints the exact numbers so the sentence above can be checked against this run.

In [10]:
s = folds[folds.model == chosen]
w04 = folds[folds.model == "W04 rule"]
print(f"{chosen}: precision@20 {s.p20.mean():.3f} ± {s.p20.std():.3f} | precision@50 {s.p50.mean():.3f} | "
      f"AUC {s.auc.mean():.3f} | base rate {s.base_rate.mean():.3f}")
print(f"W04 rule:            precision@20 {w04.p20.mean():.3f} | AUC {w04.auc.mean():.3f}")

LR, honest features: precision@20 0.828 ± 0.091 | precision@50 0.818 | AUC 0.616 | base rate 0.633
W04 rule:            precision@20 0.582 | AUC 0.522


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.